# DQN Reward Weight Search

Finds optimal reward weights using **Bayesian optimisation** (Optuna TPE).  
Training: `data/beginner_train.npz` — Evaluation: `data/beginner_test.npz` (held-out).  
Action space: `0..80` = reveal, `81..161` = flag/unflag.  
`mine_penalty` is fixed at −10 as the anchor; all other weights are searched.

**Flag reward design:** flags yield no immediate reward — correct-flag and wrong-flag signals
are realised only at episode termination, preventing any reward loop from toggling flags.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "matplotlib", "imageio", "optuna", "tqdm"],
    check=True
)

sys.path.insert(0, "..")

import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from dataclasses import dataclass, asdict
from tqdm.notebook import tqdm
from IPython.display import Image as IPImage, display

from src.env.minesweeper_env import MinesweeperEnv
from src.train.dqn import DQNAgent

print(f"torch {torch.__version__}  |  optuna {optuna.__version__}")
print("All imports OK")

## Reward Config & Search Space

In [ ]:
@dataclass
class RewardConfig:
    mine_penalty:        float = -10.0  # fixed anchor — do not search
    reveal_reward:       float =   1.0  # per newly revealed safe cell
    win_bonus:           float =  50.0  # on winning
    step_reward:         float =  -0.05 # per step (negative = time cost, discourages stalling)
    correct_flag_reward: float =   0.0  # terminal reward per correctly flagged mine
    wrong_flag_penalty:  float =   0.5  # terminal penalty per incorrectly flagged safe cell


# Search bounds for the 5 tunable weights
SEARCH_SPACE = {
    "reveal_reward":       (0.0,   2.0),
    "win_bonus":           (10.0, 100.0),
    "step_reward":         (-0.2,  0.2),  # negative half = time cost
    "correct_flag_reward": (0.0,   5.0),
    "wrong_flag_penalty":  (0.5,   5.0),  # clamped: always penalise wrong flags
}

## Reward Wrapper

In [ ]:
class RewardWrapper:
    """Wraps MinesweeperEnv: reshapes rewards per RewardConfig and adds flag actions.

    Flag actions (N..2N-1) yield no immediate reward. correct_flag_reward and
    wrong_flag_penalty are applied only at episode termination by scanning all
    flagged cells on the final board — this prevents any reward loop from
    toggling flags repeatedly mid-episode.
    """

    def __init__(self, env: MinesweeperEnv, cfg: RewardConfig):
        self._env        = env
        self.cfg         = cfg
        self.rows        = env.rows
        self.cols        = env.cols
        self.action_size = env.action_size * 2  # 0..N-1 reveal | N..2N-1 flag
        self.state_size  = env.state_size
        self._step       = 0

    def reset(self):
        self._step = 0
        return self._env.reset()

    def get_valid_actions(self):
        board   = self._env._game.board
        N       = self._env.action_size
        hidden  = board.hidden_cells()
        flagged = [i for i, v in enumerate(board.view.flatten()) if v == 9]
        return hidden + [i + N for i in hidden + flagged]

    def step(self, action):
        N     = self._env.action_size
        board = self._env._game.board
        self._step += 1

        if action >= N:
            # ---- Flag / unflag — no immediate reward ----
            cell     = action - N
            row, col = divmod(cell, self.cols)
            self._env._game.flag(row, col)
            return (board.get_observation(), 0.0, self._env._game.is_over,
                    {"state": self._env._game.state.name, "result": "flag",
                     "mines_left": board.mines_remaining()})

        # ---- Reveal ----
        obs_before = board.get_observation()
        obs, _, done, info = self._env.step(action)
        newly = (int(((obs >= 0) & (obs <= 8)).sum()) -
                 int(((obs_before >= 0) & (obs_before <= 8)).sum()))

        reward = 0.0
        if info["result"] == "mine":
            reward = self.cfg.mine_penalty
        elif info["result"] != "already_revealed":
            reward = newly * self.cfg.reveal_reward + self.cfg.step_reward
            if done and info["state"] == "WON":
                reward += self.cfg.win_bonus

        # Terminal flag reward: realised only at episode end.
        # Scan all flagged cells once — immune to mid-episode toggling.
        if done and board._mines_placed:
            for i, v in enumerate(board.view.flatten()):
                if v == 9:  # FLAGGED
                    r, c = divmod(i, self.cols)
                    if board.is_mine(r, c):
                        reward += self.cfg.correct_flag_reward
                    else:
                        reward -= self.cfg.wrong_flag_penalty

        return obs, reward, done, info

## Training / Evaluation Function

Three training completeness signals are tracked per trial:
- **`ReduceLROnPlateau`** — halves the learning rate when eval win rate stops improving (patience=3 checkpoints). When LR hits the floor the network has effectively converged.
- **Gradient norm** — mean L2 norm of all parameter gradients per episode. Near-zero norms mean the network has stopped learning regardless of LR.
- **Early stopping** — halts training if eval win rate hasn't improved in `ES_PATIENCE` consecutive checkpoints, saving time during the weight search.

In [ ]:
import time
import torch.optim.lr_scheduler as lr_sched

TRAIN_DATASET  = "../data/beginner_train.npz"
EVAL_DATASET   = "../data/beginner_test.npz"
MAX_EP_STEPS   = 500   # hard step cap — silent
MAX_EP_SECONDS = 15.0  # hard time cap — prints skip message then moves on

def _run_episode(env, agent, cfg, N):
    """Run one episode with caps. Returns final info dict (or None if capped)."""
    obs, done = env.reset(), False
    grad_norms = []
    n_flags = n_reveals = 0
    ep_start = time.perf_counter()

    while not done:
        if n_flags + n_reveals >= MAX_EP_STEPS or time.perf_counter() - ep_start >= MAX_EP_SECONDS:
            agent.push(obs, 0, cfg.mine_penalty, obs, True)
            return grad_norms, None

        action = agent.choose_action(obs, env.get_valid_actions())
        if action >= N:
            n_flags += 1
        else:
            n_reveals += 1

        next_obs, reward, done, info = env.step(action)
        agent.push(obs, action, reward, next_obs, done)
        result = agent.train_step()
        if result is not None:
            grad_norms.append(result[1])
        obs = next_obs

    return grad_norms, info


def _eval_episode(env, agent, cfg, N):
    """Run one greedy eval episode with caps. Returns True if won."""
    obs, done = env.reset(), False
    steps = 0
    ep_start = time.perf_counter()

    while not done:
        if steps >= MAX_EP_STEPS or time.perf_counter() - ep_start >= MAX_EP_SECONDS:
            return False   # treat capped eval episode as a loss
        action = agent.choose_action(obs, env.get_valid_actions())
        obs, _, done, info = env.step(action)
        steps += 1

    return info["state"] == "WON"


def run_trial(
    cfg,
    n_episodes    = 200,
    eval_every    = 50,
    eval_episodes = 20,
    model_cls     = None,
):
    """Train a fresh DQN with cfg. Returns final test win rate, eval history,
    and per-episode diagnostics (grad norms, LR, early-stop flag)."""

    _model_cls   = model_cls if model_cls is not None else MODEL_CLS
    _es_patience = ES_PATIENCE

    env   = RewardWrapper(MinesweeperEnv(preset="beginner", dataset=TRAIN_DATASET), cfg)
    agent = DQNAgent(state_size=env.state_size, action_size=env.action_size,
                     model_cls=_model_cls)

    scheduler = lr_sched.ReduceLROnPlateau(
        agent.optimiser, mode="max", factor=0.5, patience=3, min_lr=1e-5
    )

    eval_win_rates = []
    grad_norm_log  = []
    lr_log         = []

    best_eval     = 0.0
    no_improve    = 0
    stopped_early = False
    N             = env._env.action_size

    pbar = tqdm(range(n_episodes), desc="training", unit="ep", leave=False)

    for ep in pbar:
        ep_start = time.perf_counter()
        grad_norms, info = _run_episode(env, agent, cfg, N)

        elapsed = time.perf_counter() - ep_start
        if info is None:
            print(f"\n[SKIP ep={ep}] {elapsed:.1f}s — force-terminated")

        grad_norm_log.append(float(np.mean(grad_norms)) if grad_norms else float("nan"))

        pbar.set_postfix(
            wr=f"{best_eval:.3f}",
            lr=f"{agent.optimiser.param_groups[0]['lr']:.1e}",
        )

        if (ep + 1) % eval_every == 0:
            pbar.set_description("evaluating")
            saved = agent.epsilon
            agent.epsilon = 0.0
            eval_env = RewardWrapper(
                MinesweeperEnv(preset="beginner", dataset=EVAL_DATASET), cfg
            )
            wins = sum(
                _eval_episode(eval_env, agent, cfg, N)
                for _ in range(eval_episodes)
            )
            agent.epsilon = saved

            wr = wins / eval_episodes
            eval_win_rates.append(wr)
            current_lr = agent.optimiser.param_groups[0]["lr"]
            lr_log.append(current_lr)
            scheduler.step(wr)

            if wr > best_eval:
                best_eval  = wr
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= _es_patience:
                    stopped_early = True
                    pbar.set_description("early stop")
                    pbar.close()
                    break

            pbar.set_description("training")

    if not stopped_early:
        pbar.close()

    final_wr = eval_win_rates[-1] if eval_win_rates else 0.0
    return final_wr, eval_win_rates, agent, {
        "grad_norm_log": grad_norm_log,
        "lr_log":        lr_log,
        "stopped_early": stopped_early,
        "episodes_run":  (ep + 1),
    }

## Search Settings

Defaults are tuned for **< 5 min total** on CPU (both search methods combined).  
Scale up `N_TRIALS` / `N_EPISODES` once you're happy with the setup.

In [ ]:
from src.model.dqn_model import DQNModel, CNNDQNModel

# --- Model ---
MODEL_CLS  = CNNDQNModel   # swap to DQNModel to compare against flat MLP baseline

# --- Search ---
N_TRIALS   = 10    # trials per search method
N_EPISODES = 200   # training episodes per trial

# --- Early stopping ---
ES_PATIENCE = 3    # halt trial if no improvement for this many eval rounds

## Episode Trace (Quick Test)

Run `trace_episode` on an untrained agent to verify the wrapper is wired correctly before launching the full search:
- flag steps must show `r=+0.00`
- the terminal step carries the bundled flag reward
- `valid_actions` must shrink as cells are revealed

In [ ]:
def trace_episode(cfg=None, n_flag_toggles=3, seed=0):
    """
    Run one episode with a random policy and print every step.
    Deliberately toggles a flag n_flag_toggles times on the same cell to
    confirm there is no reward loop.

    Checks asserted:
      - all flag steps return reward == 0.0
      - terminal step reward includes correct_flag_reward * correctly_flagged_mines
        and subtracts wrong_flag_penalty * incorrectly_flagged_safe_cells

    Note: valid_actions can grow when a cell is unflagged (FLAGGED→HIDDEN returns
    it to both the reveal and flag halves of the action space). This is expected.
    """
    if cfg is None:
        cfg = RewardConfig(correct_flag_reward=2.0, wrong_flag_penalty=1.0)

    env   = RewardWrapper(MinesweeperEnv(preset="beginner", seed=seed), cfg)
    agent = DQNAgent(state_size=env.state_size, action_size=env.action_size,
                     model_cls=MODEL_CLS)
    agent.epsilon = 1.0   # fully random — untrained

    obs  = env.reset()
    done = False
    N    = env._env.action_size

    print(f"cfg: correct_flag_reward={cfg.correct_flag_reward}  "
          f"wrong_flag_penalty={cfg.wrong_flag_penalty}\n")
    print(f"{'step':>4}  {'type':>6}  {'cell':>8}  {'reward':>7}  "
          f"{'result':<16}  valid_actions")
    print("-" * 64)

    # Force a few flag toggles on the same cell before normal play
    board  = env._env._game.board
    target = board.hidden_cells()[0]
    for i in range(n_flag_toggles):
        flag_action = target + N
        obs, r, done, info = env.step(flag_action)
        row, col = divmod(target, env.cols)
        assert r == 0.0, f"Flag toggle gave non-zero reward: {r}"
        n_valid = len(env.get_valid_actions())
        print(f"{i+1:>4}  {'FLAG':>6}  ({row},{col})  {r:>+7.2f}  "
              f"{'flag (toggle)':<16}  {n_valid}")

    step = n_flag_toggles
    while not done:
        valid  = env.get_valid_actions()
        action = agent.choose_action(obs, valid)
        obs, r, done, info = env.step(action)
        step += 1

        kind = "FLAG" if action >= N else "REVEAL"
        cell_idx = action % N
        row, col = divmod(cell_idx, env.cols)
        if kind == "FLAG":
            assert r == 0.0, f"Flag action gave non-zero mid-episode reward: {r}"

        suffix = ""
        if done:
            board = env._env._game.board
            if board._mines_placed:
                correct_flags = [i for i, v in enumerate(board.view.flatten())
                                 if v == 9 and board.is_mine(*divmod(i, env.cols))]
                wrong_flags   = [i for i, v in enumerate(board.view.flatten())
                                 if v == 9 and not board.is_mine(*divmod(i, env.cols))]
                flag_component = (len(correct_flags) * cfg.correct_flag_reward
                                  - len(wrong_flags)  * cfg.wrong_flag_penalty)
                base_r = r - flag_component
                suffix = (f"  ← terminal  "
                          f"base={base_r:+.2f}  "
                          f"flags: {len(correct_flags)} correct (+{len(correct_flags)*cfg.correct_flag_reward:.2f})"
                          f"  {len(wrong_flags)} wrong (-{len(wrong_flags)*cfg.wrong_flag_penalty:.2f})"
                          f"  total={r:+.2f}")

        print(f"{step:>4}  {kind:>6}  ({row},{col})  {r:>+7.2f}  "
              f"{info['result']:<16}  {len(env.get_valid_actions())}{suffix}")

    print("-" * 64)
    print(f"Outcome: {info['state']}")
    print("All flag-reward assertions passed ✓")


trace_episode()

## Bayesian Search (Optuna TPE)

In [ ]:
bayes_trials      = []
bayes_best_so_far = []

def objective(trial):
    cfg = RewardConfig(
        reveal_reward       = trial.suggest_float("reveal_reward",       *SEARCH_SPACE["reveal_reward"]),
        win_bonus           = trial.suggest_float("win_bonus",           *SEARCH_SPACE["win_bonus"]),
        step_reward         = trial.suggest_float("step_reward",         *SEARCH_SPACE["step_reward"]),
        correct_flag_reward = trial.suggest_float("correct_flag_reward", *SEARCH_SPACE["correct_flag_reward"]),
        wrong_flag_penalty  = trial.suggest_float("wrong_flag_penalty",  *SEARCH_SPACE["wrong_flag_penalty"]),
    )
    final_wr, history, _, diag = run_trial(cfg, n_episodes=N_EPISODES)

    bayes_trials.append({"cfg": cfg, "final_wr": final_wr, "history": history, "diag": diag})
    best_so_far = max(r["final_wr"] for r in bayes_trials)
    bayes_best_so_far.append(best_so_far)
    t = len(bayes_trials)
    flag = " [early stop]" if diag["stopped_early"] else ""
    print(f"  [bayes]  trial {t:>3}/{N_TRIALS}  "
          f"wr={final_wr:.3f}  best={best_so_far:.3f}  "
          f"eps={diag['episodes_run']:>6}  "
          f"lr={diag['lr_log'][-1]:.2e}{flag}")
    return final_wr

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS)

best_bayes = max(bayes_trials, key=lambda r: r["final_wr"])
print(f"\nBayesian search best: {best_bayes['final_wr']:.3f}")

## Results

In [ ]:
os.makedirs("../recordings", exist_ok=True)

all_bayes_wrs = [r["final_wr"] for r in bayes_trials]
weight_keys   = list(SEARCH_SPACE.keys())

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Bayesian Optimisation (Optuna TPE) — DQN Reward Weights", fontsize=14)

# ---- 1. Best-so-far convergence ----
ax = fig.add_subplot(2, 3, 1)
ax.plot(range(1, N_TRIALS + 1), [v * 100 for v in bayes_best_so_far],
        color="tomato", lw=2, marker="s", ms=4, label="Bayesian (TPE)")
ax.set_title("Best-So-Far Test Win Rate")
ax.set_xlabel("Trial")
ax.set_ylabel("Win rate (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- 2. All trial win rates (scatter) ----
ax = fig.add_subplot(2, 3, 2)
ax.scatter(range(1, N_TRIALS + 1), [v * 100 for v in all_bayes_wrs],
           color="tomato", alpha=0.7, label="Bayesian", marker="s", zorder=3)
ax.set_title("All Trial Win Rates")
ax.set_xlabel("Trial")
ax.set_ylabel("Test win rate (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- 3. Distribution (box plot) ----
ax = fig.add_subplot(2, 3, 3)
bp = ax.boxplot(
    [[v * 100 for v in all_bayes_wrs]],
    labels=["Bayesian"],
    patch_artist=True,
    medianprops={"color": "black", "lw": 2},
)
bp["boxes"][0].set_facecolor("tomato")
bp["boxes"][0].set_alpha(0.6)
ax.set_title("Win Rate Distribution")
ax.set_ylabel("Test win rate (%)")
ax.grid(True, alpha=0.3, axis="y")

# ---- 4–6. Weight vs win rate scatter (top 3 most important weights) ----
for idx, key in enumerate(weight_keys[:3]):
    ax = fig.add_subplot(2, 3, 4 + idx)
    bayes_x = [r["cfg"].__dict__[key] for r in bayes_trials]
    ax.scatter(bayes_x, [v * 100 for v in all_bayes_wrs],
               color="tomato", alpha=0.7, label="Bayesian", marker="s", zorder=3)
    lo, hi = SEARCH_SPACE[key]
    ax.set_xlim(lo, hi)
    ax.set_title(f"{key} vs win rate")
    ax.set_xlabel(key)
    ax.set_ylabel("Test win rate (%)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/dqn_ablation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/dqn_ablation.png")

## Training Completeness Diagnostics (Best Config)

In [ ]:
# Use best Bayesian config's diagnostics
diag = best_bayes["diag"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Training Completeness — Best Bayesian Config", fontsize=13)

# ---- 1. Gradient norm over training ----
ax = axes[0]
gnorms = [g for g in diag["grad_norm_log"] if not np.isnan(g)]
window = min(100, len(gnorms) // 5 or 1)
smoothed = np.convolve(gnorms, np.ones(window) / window, mode="valid")
ax.plot(smoothed, color="steelblue", lw=1)
ax.set_title("Gradient Norm (smoothed)")
ax.set_xlabel("Episode")
ax.set_ylabel("L2 norm")
ax.grid(True, alpha=0.3)
ax.axhline(0.01, color="tomato", lw=1, ls="--", label="Near-zero threshold")
ax.legend(fontsize=8)

# ---- 2. Learning rate schedule ----
ax = axes[1]
eval_xs = [500 * (i + 1) for i in range(len(diag["lr_log"]))]
ax.plot(eval_xs, diag["lr_log"], color="darkorange", lw=2, marker="o", ms=4)
ax.set_title("Learning Rate (ReduceLROnPlateau)")
ax.set_xlabel("Episode")
ax.set_ylabel("LR")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
lrs = diag["lr_log"]
for i in range(1, len(lrs)):
    if lrs[i] < lrs[i - 1]:
        ax.axvline(eval_xs[i], color="tomato", lw=1, ls="--", alpha=0.6)

# ---- 3. Early stopping across Bayesian trials ----
ax = axes[2]
bayes_eps     = [r["diag"]["episodes_run"] for r in bayes_trials]
bayes_stopped = sum(r["diag"]["stopped_early"] for r in bayes_trials)
ax.hist(bayes_eps, bins=10, alpha=0.7, color="tomato",
        label=f"Bayesian ({bayes_stopped}/{N_TRIALS} early stopped)")
ax.axvline(N_EPISODES, color="black", lw=1.2, ls="--", label=f"Max ({N_EPISODES})")
ax.set_title("Episodes Run per Trial")
ax.set_xlabel("Episodes")
ax.set_ylabel("Count")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../recordings/training_completeness.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: recordings/training_completeness.png")
print(f"\nBest config ran {diag['episodes_run']} / {N_EPISODES} episodes  "
      f"({'early stopped' if diag['stopped_early'] else 'ran to completion'})")
print(f"Final LR: {diag['lr_log'][-1]:.2e}  |  "
      f"Final mean grad norm: {np.nanmean(diag['grad_norm_log'][-500:]):.4f}")

## Best Configs Summary

In [ ]:
print("=" * 56)
print("  BAYESIAN SEARCH — best config")
print("=" * 56)
print(f"  Test win rate:       {best_bayes['final_wr']*100:.2f}%")
for k in weight_keys:
    print(f"  {k:<24} {best_bayes['cfg'].__dict__[k]:.4f}")

print()
print("=" * 56)
print("  ALL BAYESIAN TRIALS")
print("=" * 56)
print(f"  mean: {np.mean(all_bayes_wrs)*100:.2f}%  "
      f"max: {max(all_bayes_wrs)*100:.2f}%  "
      f"std: {np.std(all_bayes_wrs)*100:.2f}%")

## GIF — Best Config

In [ ]:
cfg = RewardConfig()
wr, history, _, diag = run_trial(cfg, n_episodes=10, eval_every=10, eval_episodes=5)
print(f"win rate: {wr:.3f}  |  episodes: {diag['episodes_run']}")

In [ ]:
import pygame
pygame.init()
from src.game.renderer import Renderer

print(f"Recording episode using Bayesian best config "
      f"(test wr={best_bayes['final_wr']*100:.1f}%)")

# Re-train a fresh agent with the best config for recording
_, _, rec_agent, _ = run_trial(best_bayes["cfg"], n_episodes=N_EPISODES)
rec_agent.epsilon = 0.0  # greedy

rec_base = MinesweeperEnv(preset="beginner", dataset=EVAL_DATASET)
rec_env  = RewardWrapper(rec_base, best_bayes["cfg"])
obs      = rec_env.reset()

rec_renderer = Renderer(rec_base._game, preset="beginner")

def capture():
    rec_renderer._draw()
    return pygame.surfarray.array3d(rec_renderer.screen).transpose(1, 0, 2).copy()

frames = [capture()]
done   = False
while not done:
    action = rec_agent.choose_action(obs, rec_env.get_valid_actions())
    obs, _, done, info = rec_env.step(action)
    frames.append(capture())

GIF_PATH = "../recordings/dqn_best_episode.gif"
imageio.mimsave(GIF_PATH, frames, duration=0.4, loop=0)
print(f"Outcome: {info['state']} in {len(frames)-1} steps")
print(f"Saved: {GIF_PATH}")
display(IPImage(filename=GIF_PATH))